In [12]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
import cohere
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

True

In [2]:
loader = PyPDFLoader('AI_Engineering_Roadmap_2026.pdf')

documents = loader.load()

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)

chunks = splitter.split_documents(documents=documents)

In [7]:
len(chunks)

68

In [8]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma.from_documents(documents=chunks, embedding=embedding, persist_directory="./vectorstore")

W1031 01:08:39.897000 98473 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
co = cohere.ClientV2(api_key=os.getenv('COHERE_API_KEY'))

def re_rank_documents(query: str, documents: list[Document], top_n: int = 3):
    if not documents:
        return []

    document_texts = [doc.page_content for doc in documents]

    response = co.rerank(
        model="rerank-v5.0-fast",
        query=query,
        documents=document_texts,
        top_n=min(top_n, len(documents))
    )

    reranked_documents = []

    for result in response.results:
        original_doc = documents[result.index]

        reranked_documents.append({
            "document": original_doc,
            "score": result.relevance_score
        })

    return reranked_documents

In [13]:
llm = ChatGroq(
    model=os.getenv("GROQ_MODEL"),
    temperature=0
)

# 4. Define RAG prompt
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the question using ONLY the
provided context.

If the answer is not available in the
context, say that you don't know.

Context:
{context}

Question:
{question}

Answer:
""")

rag_chain = prompt | llm | StrOutputParser()

def ask_question(question: str):
     retrieved_docs = vector_store.similarity_search(query=question, k=10)

     reranked_docs = re_rank_documents(question, retrieved_docs, 2)

     print("Reranked Docs: ", reranked_docs)

     context = "\n\n".join(
        item["document"].page_content
        for item in reranked_docs
     )

     answer = rag_chain.invoke({
        "context": context,
        "question": question
     })

     return answer

In [15]:
res = ask_question("RAG Topics to cover")

print(res)

Reranked Docs:  [{'document': Document(metadata={'total_pages': 11, 'page_label': '6', 'trapped': '/False', 'subject': '(unspecified)', 'keywords': '', 'source': 'AI_Engineering_Roadmap_2026.pdf', 'producer': 'ReportLab PDF Library - (opensource)', 'author': '(anonymous)', 'title': 'AI Engineering: Complete Step-by-Step Roadmap (2026)', 'page': 5, 'moddate': '2026-10-12T20:05:10+00:00', 'creator': '(unspecified)', 'creationdate': '2026-10-12T20:05:10+00:00'}, page_content='Provide retrieved context to an LLM and return citations\nTopics to learn\n \x7f Embedding models and cosine similarity.\n\x7f Chunk size, overlap and metadata.\n\x7f Vector search vs. keyword search.\n\x7f Hybrid search.\n\x7f Metadata filtering.\n\x7f Reranking.\n\x7f Context construction.\n\x7f Retrieval precision and recall.\n\x7f RAG evaluation and hallucination reduction.\n\x7f Document access controls and source citations.'), 'score': 0.8075287}, {'document': Document(metadata={'title': 'AI Engineering: Comple